#### Assignment 8: Support Vector Machine

In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd

url = "https://users.stat.ufl.edu/~winner/data/armada.dat"

# Column names
column_names = [
    "Battle", "Year", "Portuguese_ships", "Dutch_ships", 
    "English_ships", "Ratio_Pt_Dutch_British", "Spanish_involvement", "Portuguese_outcome"
]

df = pd.read_fwf(url, header=None, names=column_names)

print(df.head())



           Battle  Year  Portuguese_ships  Dutch_ships  English_ships  \
0          Bantam  1601                 6            3              0   
1  Malacca Strait  1606                14           11              0   
2   Ilha das Naus  1606                 6            9              0   
3      Pulo Butum  1606                 7            9              0   
4          Surrat  1615                 6            0              4   

   Ratio_Pt_Dutch_British  Spanish_involvement  Portuguese_outcome  
0                   2.000                    0                   0  
1                   1.273                    0                   0  
2                   0.667                    0                  -1  
3                   0.778                    0                   1  
4                   1.500                    0                   0  


In [3]:
#data inspection

In [4]:
df.head()

,Battle,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_Pt_Dutch_British,Spanish_involvement,Portuguese_outcome
0,Bantam,1601,6,3,0,2.000,0,0
1,Malacca Strait,1606,14,11,0,1.273,0,0
2,Ilha das Naus,1606,6,9,0,0.667,0,-1
3,Pulo Butum,1606,7,9,0,0.778,0,1
4,Surrat,1615,6,0,4,1.500,0,0


In [5]:

print(df.isnull().sum())

print(df.dtypes)


Battle                    0
Year                      0
Portuguese_ships          0
Dutch_ships               0
English_ships             0
Ratio_Pt_Dutch_British    0
Spanish_involvement       0
Portuguese_outcome        0
dtype: int64
Battle                     object
Year                        int64
Portuguese_ships            int64
Dutch_ships                 int64
English_ships               int64
Ratio_Pt_Dutch_British    float64
Spanish_involvement         int64
Portuguese_outcome          int64
dtype: object


In [6]:
df.describe()


,Year,Portuguese_ships,Dutch_ships,English_ships,Ratio_Pt_Dutch_British,Spanish_involvement,Portuguese_outcome
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,1628.392857,13.142857,13.428571,1.785714,1.159893,0.464286,-0.178571
std,17.559084,15.922763,22.280083,5.927695,0.928341,0.507875,0.722832
min,1588.000000,2.000000,0.000000,0.000000,0.150000,0.000000,-1.000000
25%,1618.750000,5.750000,4.000000,0.000000,0.650250,0.000000,-1.000000
50%,1628.500000,6.000000,8.000000,0.000000,0.928500,0.000000,0.000000
75%,1639.000000,14.000000,11.000000,0.000000,1.500000,1.000000,0.000000
max,1658.000000,69.000000,110.000000,31.000000,4.636000,1.000000,1.000000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Battle                  28 non-null     object 
 1   Year                    28 non-null     int64  
 2   Portuguese_ships        28 non-null     int64  
 3   Dutch_ships             28 non-null     int64  
 4   English_ships           28 non-null     int64  
 5   Ratio_Pt_Dutch_British  28 non-null     float64
 6   Spanish_involvement     28 non-null     int64  
 7   Portuguese_outcome      28 non-null     int64  
dtypes: float64(1), int64(6), object(1)
memory usage: 1.9+ KB


In [8]:
#Select Features and Target 

features = df[['Portuguese_ships', 'Dutch_ships', 'English_ships', 
               'Ratio_Pt_Dutch_British', 'Spanish_involvement']]
target = df['Portuguese_outcome']

In [9]:
# Splitting Data into Training and Test Sets 75% training, 25% testing, with stratified split 
features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.25, random_state=42, stratify=target
)

In [10]:
#SVM model

# scalling features using StandardScaler and using Linear kernel with class weighting 
svm_model = make_pipeline(
    StandardScaler(), 
    SVC(kernel='linear', class_weight='balanced', random_state=42)  
)

#Model training 
svm_model.fit(features_train, target_train)

# predicting on training and testing sets
train_predictions = svm_model.predict(features_train)  
test_predictions = svm_model.predict(features_test)   

# calculating and storing Training accuracy and Test accuracy
svm_train_accuracy = accuracy_score(target_train, train_predictions)  
svm_test_accuracy= accuracy_score(target_test, test_predictions)     


#output
print(" SVM Model ")
print(f"Train Accuracy: {svm_train_accuracy:.2f}")
print(f"Test Accuracy: {svm_test_accuracy:.2f}")    
print("\nClassification Report (Test):\n", classification_report(target_test, test_predictions, zero_division=0)) 


 SVM Model 
Train Accuracy: 0.52
Test Accuracy: 0.57

Classification Report (Test):
               precision    recall  f1-score   support

          -1       0.60      1.00      0.75         3
           0       0.00      0.00      0.00         3
           1       0.50      1.00      0.67         1

    accuracy                           0.57         7
   macro avg       0.37      0.67      0.47         7
weighted avg       0.33      0.57      0.42         7



The SVM model got 52% accuracy on training data and 57% on test data, meaning it’s correct about half the time. For the test set:

Class -1(Defeat): it predicted all 3 correctly (recall 100%), but had some  wrong predictions (precision 60%).

Class 0 (Draw): it didn’t predict any correctly (precision 0%, recall 0%).

Class 1(victory): the one instance was correctly predicted (recall 100%), but there was one wrong prediction (precision 50%).

Overall, the model is better at detecting -1 and 1, but struggles with 0.

In [11]:
# Decision Tree Model 
from sklearn.tree import DecisionTreeClassifier

# Initializing Decision Tree with max_depth to prevent overfitting
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)

#Model training 
dt_model.fit(features_train, target_train)

# predicting on training and testing sets
dt_train_pred = dt_model.predict(features_train)
dt_test_pred = dt_model.predict(features_test)

# calculating and storing Training accuracy and Test accuracy
dt_train_accuracy = accuracy_score(target_train, dt_train_pred)  
dt_test_accuracy= accuracy_score(target_test, dt_test_pred)     

#output
print("Decision Tree Model ")
print(f"Train Accuracy:{dt_train_accuracy:.2f}")  
print(f"Test Accuracy: {dt_test_accuracy:.2f}")    
print("\nClassification Report (Test):\n", classification_report(target_test, dt_test_pred, zero_division=0))  # Detailed metrics



Decision Tree Model 
Train Accuracy:0.76
Test Accuracy: 0.71

Classification Report (Test):
               precision    recall  f1-score   support

          -1       1.00      0.67      0.80         3
           0       0.60      1.00      0.75         3
           1       0.00      0.00      0.00         1

    accuracy                           0.71         7
   macro avg       0.53      0.56      0.52         7
weighted avg       0.69      0.71      0.66         7



The Decision Tree model got 76% accuracy on training data and 71% on test data, meaning it’s correct most of the time. For the test set:

Class -1: it predicted 2 out of 3 correctly (recall 67%), and out of all predictions for -1, all were correct (precision 100%).

Class 0: it predicted all 3 correctly (recall 100%), but some predictions were wrong (precision 60%).

Class 1: it didn’t predict the single instance correctly (precision 0%, recall 0%).

Overall, the model performs well for -1 and 0, but struggles with class 1, and it generalizes better than Random Forest without overfitting too much.

In [12]:
# Random Forest Model
from sklearn.ensemble import RandomForestClassifier

# Initializing Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

#Model training 
rf_model.fit(features_train, target_train)

# predicting on training and testing sets
train_pred = rf_model.predict(features_train)
test_pred = rf_model.predict(features_test)

# calculating and storing Training accuracy and Test accuracy

rf_train_accuracy = accuracy_score(target_train, train_pred)
rf_test_accuracy = accuracy_score(target_test, test_pred)

#output
print("Random Forest Results")
print(f"Train Accuracy: {rf_train_accuracy:.2f}")
print(f"Test Accuracy:{rf_test_accuracy:.2f}")
print("Classification Report (Test):\n", classification_report(target_test, test_pred, zero_division=0))


Random Forest Results
Train Accuracy: 1.00
Test Accuracy:0.29
Classification Report (Test):
               precision    recall  f1-score   support

          -1       0.50      0.33      0.40         3
           0       0.25      0.33      0.29         3
           1       0.00      0.00      0.00         1

    accuracy                           0.29         7
   macro avg       0.25      0.22      0.23         7
weighted avg       0.32      0.29      0.29         7



The Random Forest model got 100% accuracy on training data but only 29% on test data, showing it doesn’t generalize well.

For the test set:

Class -1: it predicted some correctly, but not all.

Class 0: it predicted only a few correctly.

Class 1: it didn’t predict correctly at all.

Overall, the model overfits and struggles to predict new data accurately

In [13]:
# --- Final Comparison of Models ---

print("-->Final Model Comparison\n")

print(f"SVM:            Train Accuracy = {svm_train_accuracy:.2f}, Test Accuracy = {svm_test_accuracy:.2f}")
print(f"Decision Tree:  Train Accuracy = {dt_train_accuracy:.2f}, Test Accuracy = {dt_test_accuracy:.2f}")
print(f"Random Forest:  Train Accuracy = {rf_train_accuracy:.2f}, Test Accuracy = {rf_test_accuracy:.2f}")



-->Final Model Comparison

SVM:            Train Accuracy = 0.52, Test Accuracy = 0.57
Decision Tree:  Train Accuracy = 0.76, Test Accuracy = 0.71
Random Forest:  Train Accuracy = 1.00, Test Accuracy = 0.29


The SVM model achieved 52% training accuracy and 57% test accuracy, showing moderate performance. The Decision Tree did better, with 76% training and 71% test accuracy, learning patterns more effectively and generalizing well. The Random Forest overfitted the data, with 100% training accuracy but only 29% test accuracy. Overall, the Decision Tree was the most balanced and reliable model for this dataset.